# 1. Librerías 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf

from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.stats.diagnostic import acorr_ljungbox

import warnings
warnings.filterwarnings("ignore")

# 2. Carga de datos

In [2]:
df_corr = pd.read_excel(r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\datos_consolidados\3_datos_modelos\datos_corr\datos_modelo_1_var_rezago_relevante\datos_modelo_1_var_rezago_relevante.xlsx")

df_corr['fecha'] = pd.to_datetime(df_corr['fecha'])
df_corr = df_corr.sort_values('fecha')
df_corr.set_index('fecha', inplace=True)
df_corr.columns

Index(['año', 'semana_epi', 'casos_ln', 'hum_esp_bc_lag_6',
       'vel_vi_max_bc_lag_5', 'prec_ln_lag_8', 'temp_max_bc_lag_7',
       'sst_yj_lag_20'],
      dtype='object')

# 3. Rutas

In [3]:
ubicacion_datos_consolidados_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\datos_consolidados\3_datos_modelos\datos_corr\datos_modelo_1_var_rezago_relevante\resultados"

ubicacion_imagenes_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\datos_consolidados\3_datos_modelos\datos_corr\datos_modelo_1_var_rezago_relevante\imagenes"

os.makedirs(ubicacion_datos_consolidados_janis, exist_ok=True)
os.makedirs(ubicacion_imagenes_janis, exist_ok=True)

In [4]:
# ============================================================
# VARIABLE OBJETIVO
# ============================================================

y = df_corr['casos_ln']


# ============================================================
# VARIABLES EXÓGENAS
# ============================================================

exog = df_corr[
    [
        'hum_esp_bc_lag_6',
       'vel_vi_max_bc_lag_5', 'prec_ln_lag_8', 'temp_max_bc_lag_7',
       'sst_yj_lag_20'
    ]
]

In [5]:
# ============================================================
# DIVISIÓN TRAIN TEST
# ============================================================

# ------------------------------------------------------------
# 80 - 20
# ------------------------------------------------------------

split_80 = int(len(df_corr) * 0.80)

y_train_80 = y.iloc[:split_80]
y_test_80 = y.iloc[split_80:]

X_train_80 = exog.iloc[:split_80]
X_test_80 = exog.iloc[split_80:]


# ------------------------------------------------------------
# 90 - 10
# ------------------------------------------------------------

split_90 = int(len(df_corr) * 0.90)

y_train_90 = y.iloc[:split_90]
y_test_90 = y.iloc[split_90:]

X_train_90 = exog.iloc[:split_90]
X_test_90 = exog.iloc[split_90:]


# ------------------------------------------------------------
# 95 - 5
# ------------------------------------------------------------

split_95 = int(len(df_corr) * 0.95)

y_train_95 = y.iloc[:split_95]
y_test_95 = y.iloc[split_95:]

X_train_95 = exog.iloc[:split_95]
X_test_95 = exog.iloc[split_95:]

In [6]:
# ============================================================
# FUNCIÓN MAPE
# ============================================================

def mape(y_true, y_pred):

    return np.mean(
        np.abs((y_true - y_pred) / y_true)
    ) * 100


# ============================================================
# FUNCIÓN EVALUACIÓN
# ============================================================

def evaluar_modelo(modelo_fit, y_test, predicciones):

    rmse = np.sqrt(
        mean_squared_error(y_test, predicciones)
    )

    mae = mean_absolute_error(
        y_test,
        predicciones
    )

    mape_value = mape(
        y_test,
        predicciones
    )

    ljung = acorr_ljungbox(
        modelo_fit.resid,
        lags=[10],
        return_df=True
    )

    pvalue_ljung = ljung['lb_pvalue'].values[0]

    resultados = {

        'AIC': modelo_fit.aic,
        'BIC': modelo_fit.bic,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape_value,
        'Ljung_Box_pvalue': pvalue_ljung

    }

    return resultados


In [7]:
# ============================================================
# FUNCIÓN PARA GRAFICAR
# ============================================================

def graficar_predicciones(
    y_test,
    predicciones,
    nombre_modelo,
    division
):

    plt.figure(figsize=(12,6))

    plt.plot(
        y_test.index,
        y_test,
        label='Observado'
    )

    plt.plot(
        predicciones.index,
        predicciones,
        label='Predicción'
    )

    plt.title(
        f'{nombre_modelo} - División {division}'
    )

    plt.xlabel('Fecha')
    plt.ylabel('Casos dengue')

    plt.legend()

    nombre_imagen = (
        f'{nombre_modelo}_{division}.png'
    )

    ruta_imagen = os.path.join(
        ubicacion_imagenes_janis,
        nombre_imagen
    )

    plt.savefig(
        ruta_imagen,
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

In [8]:
# ============================================================
# MODELOS CANDIDATOS ARIMA
# ============================================================

modelos_arima = [

    (1,1,0),
    (0,1,1),
    (1,1,1),
    (2,1,1),
    (1,1,2),
    (2,1,2)

]


# ============================================================
# MODELOS CANDIDATOS SARIMA
# ============================================================

modelos_sarima = [

    ((1,1,1),(1,1,1,52)),
    ((1,1,0),(1,1,1,52)),
    ((0,1,1),(1,1,1,52)),
    ((2,1,1),(1,1,1,52)),
    ((1,1,2),(1,1,1,52)),
    ((1,1,1),(0,1,1,52)),
    ((1,1,1),(1,1,0,52))

]

In [9]:
# ============================================================
# FUNCIÓN ARIMA
# ============================================================

def ejecutar_arima(
    modelos,
    y_train,
    y_test,
    division
):

    resultados = []

    for order in modelos:

        try:

            modelo = ARIMA(
                y_train,
                order=order
            )

            modelo_fit = modelo.fit()

            predicciones = modelo_fit.forecast(
                steps=len(y_test)
            )

            metricas = evaluar_modelo(
                modelo_fit,
                y_test,
                predicciones
            )

            nombre_modelo = f'ARIMA{order}'

            resultados.append({
                'Modelo': nombre_modelo,
                **metricas
            })

            # -----------------------------------
            # Guardar gráfico
            # -----------------------------------

            graficar_predicciones(
                y_test,
                predicciones,
                nombre_modelo,
                division
            )

            print(f'{nombre_modelo} OK')

        except Exception as e:

            print(f'Error {order}: {e}')

    return pd.DataFrame(resultados)



In [10]:
# ============================================================
# FUNCIÓN SARIMA
# ============================================================

def ejecutar_sarima(
    modelos,
    y_train,
    y_test,
    division
):

    resultados = []

    for order, seasonal_order in modelos:

        try:

            modelo = SARIMAX(
                y_train,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            modelo_fit = modelo.fit(disp=False)

            predicciones = modelo_fit.forecast(
                steps=len(y_test)
            )

            metricas = evaluar_modelo(
                modelo_fit,
                y_test,
                predicciones
            )

            nombre_modelo = (
                f'SARIMA{order}{seasonal_order}'
            )

            resultados.append({
                'Modelo': nombre_modelo,
                **metricas
            })

            # -----------------------------------
            # Guardar gráfico
            # -----------------------------------

            graficar_predicciones(
                y_test,
                predicciones,
                nombre_modelo,
                division
            )

            print(f'{nombre_modelo} OK')

        except Exception as e:

            print(f'Error {order}: {e}')

    return pd.DataFrame(resultados)



In [11]:
# ============================================================
# FUNCIÓN ARIMAX
# ============================================================

def ejecutar_arimax(
    modelos,
    y_train,
    y_test,
    X_train,
    X_test,
    division
):

    resultados = []

    for order in modelos:

        try:

            modelo = SARIMAX(
                y_train,
                exog=X_train,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            modelo_fit = modelo.fit(disp=False)

            predicciones = modelo_fit.forecast(
                steps=len(y_test),
                exog=X_test
            )

            metricas = evaluar_modelo(
                modelo_fit,
                y_test,
                predicciones
            )

            nombre_modelo = f'ARIMAX{order}'

            resultados.append({
                'Modelo': nombre_modelo,
                **metricas
            })

            # -----------------------------------
            # Guardar gráfico
            # -----------------------------------

            graficar_predicciones(
                y_test,
                predicciones,
                nombre_modelo,
                division
            )

            print(f'{nombre_modelo} OK')

        except Exception as e:

            print(f'Error {order}: {e}')

    return pd.DataFrame(resultados)



In [12]:
# ============================================================
# FUNCIÓN SARIMAX
# ============================================================

def ejecutar_sarimax(
    modelos,
    y_train,
    y_test,
    X_train,
    X_test,
    division
):

    resultados = []

    for order, seasonal_order in modelos:

        try:

            modelo = SARIMAX(
                y_train,
                exog=X_train,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            modelo_fit = modelo.fit(disp=False)

            predicciones = modelo_fit.forecast(
                steps=len(y_test),
                exog=X_test
            )

            metricas = evaluar_modelo(
                modelo_fit,
                y_test,
                predicciones
            )

            nombre_modelo = (
                f'SARIMAX{order}{seasonal_order}'
            )

            resultados.append({
                'Modelo': nombre_modelo,
                **metricas
            })

            # -----------------------------------
            # Guardar gráfico
            # -----------------------------------

            graficar_predicciones(
                y_test,
                predicciones,
                nombre_modelo,
                division
            )

            print(f'{nombre_modelo} OK')

        except Exception as e:

            print(f'Error {order}: {e}')

    return pd.DataFrame(resultados)



In [13]:
# ============================================================
# IMPLEMENTACIÓN 80 - 20
# ============================================================

resultados_arima_80 = ejecutar_arima(
    modelos_arima,
    y_train_80,
    y_test_80,
    division='80_20'
)

resultados_sarima_80 = ejecutar_sarima(
    modelos_sarima,
    y_train_80,
    y_test_80,
    division='80_20'
)

resultados_arimax_80 = ejecutar_arimax(
    modelos_arima,
    y_train_80,
    y_test_80,
    X_train_80,
    X_test_80,
    division='80_20' 
)

resultados_sarimax_80 = ejecutar_sarimax(
    modelos_sarima,
    y_train_80,
    y_test_80,
    X_train_80,
    X_test_80,
    division='80_20'
)

ARIMA(1, 1, 0) OK
ARIMA(0, 1, 1) OK
ARIMA(1, 1, 1) OK
ARIMA(2, 1, 1) OK
ARIMA(1, 1, 2) OK
ARIMA(2, 1, 2) OK
SARIMA(1, 1, 1)(1, 1, 1, 52) OK
SARIMA(1, 1, 0)(1, 1, 1, 52) OK
SARIMA(0, 1, 1)(1, 1, 1, 52) OK
SARIMA(2, 1, 1)(1, 1, 1, 52) OK
SARIMA(1, 1, 2)(1, 1, 1, 52) OK
SARIMA(1, 1, 1)(0, 1, 1, 52) OK
SARIMA(1, 1, 1)(1, 1, 0, 52) OK
ARIMAX(1, 1, 0) OK
ARIMAX(0, 1, 1) OK
ARIMAX(1, 1, 1) OK
ARIMAX(2, 1, 1) OK
ARIMAX(1, 1, 2) OK
ARIMAX(2, 1, 2) OK
SARIMAX(1, 1, 1)(1, 1, 1, 52) OK
SARIMAX(1, 1, 0)(1, 1, 1, 52) OK
SARIMAX(0, 1, 1)(1, 1, 1, 52) OK
SARIMAX(2, 1, 1)(1, 1, 1, 52) OK
SARIMAX(1, 1, 2)(1, 1, 1, 52) OK
SARIMAX(1, 1, 1)(0, 1, 1, 52) OK
SARIMAX(1, 1, 1)(1, 1, 0, 52) OK


In [14]:
# ============================================================
# IMPLEMENTACIÓN 90 - 10
# ============================================================

resultados_arima_90 = ejecutar_arima(
    modelos_arima,
    y_train_90,
    y_test_90,
    division='90_10'
)

resultados_sarima_90 = ejecutar_sarima(
    modelos_sarima,
    y_train_90,
    y_test_90,
    division='90_10'
)

resultados_arimax_90 = ejecutar_arimax(
    modelos_arima,
    y_train_90,
    y_test_90,
    X_train_90,
    X_test_90,
    division='90_10'
)

resultados_sarimax_90 = ejecutar_sarimax(
    modelos_sarima,
    y_train_90,
    y_test_90,
    X_train_90,
    X_test_90,
    division='90_10'
)

ARIMA(1, 1, 0) OK
ARIMA(0, 1, 1) OK
ARIMA(1, 1, 1) OK
ARIMA(2, 1, 1) OK
ARIMA(1, 1, 2) OK
ARIMA(2, 1, 2) OK
SARIMA(1, 1, 1)(1, 1, 1, 52) OK
SARIMA(1, 1, 0)(1, 1, 1, 52) OK
SARIMA(0, 1, 1)(1, 1, 1, 52) OK
SARIMA(2, 1, 1)(1, 1, 1, 52) OK
SARIMA(1, 1, 2)(1, 1, 1, 52) OK
SARIMA(1, 1, 1)(0, 1, 1, 52) OK
SARIMA(1, 1, 1)(1, 1, 0, 52) OK
ARIMAX(1, 1, 0) OK
ARIMAX(0, 1, 1) OK
ARIMAX(1, 1, 1) OK
ARIMAX(2, 1, 1) OK
ARIMAX(1, 1, 2) OK
ARIMAX(2, 1, 2) OK
SARIMAX(1, 1, 1)(1, 1, 1, 52) OK
SARIMAX(1, 1, 0)(1, 1, 1, 52) OK
SARIMAX(0, 1, 1)(1, 1, 1, 52) OK
SARIMAX(2, 1, 1)(1, 1, 1, 52) OK
SARIMAX(1, 1, 2)(1, 1, 1, 52) OK
SARIMAX(1, 1, 1)(0, 1, 1, 52) OK
SARIMAX(1, 1, 1)(1, 1, 0, 52) OK


In [15]:
# ============================================================
# IMPLEMENTACIÓN 95 - 5
# ============================================================

resultados_arima_95 = ejecutar_arima(
    modelos_arima,
    y_train_95,
    y_test_95,
    division='95_5'
)

resultados_sarima_95 = ejecutar_sarima(
    modelos_sarima,
    y_train_95,
    y_test_95,
    division='95_5'
)

resultados_arimax_95 = ejecutar_arimax(
    modelos_arima,
    y_train_95,
    y_test_95,
    X_train_95,
    X_test_95,
    division='95_5'
)

resultados_sarimax_95 = ejecutar_sarimax(
    modelos_sarima,
    y_train_95,
    y_test_95,
    X_train_95,
    X_test_95,
    division='95_5'
)

ARIMA(1, 1, 0) OK
ARIMA(0, 1, 1) OK
ARIMA(1, 1, 1) OK
ARIMA(2, 1, 1) OK
ARIMA(1, 1, 2) OK
ARIMA(2, 1, 2) OK
SARIMA(1, 1, 1)(1, 1, 1, 52) OK
SARIMA(1, 1, 0)(1, 1, 1, 52) OK
SARIMA(0, 1, 1)(1, 1, 1, 52) OK
SARIMA(2, 1, 1)(1, 1, 1, 52) OK
SARIMA(1, 1, 2)(1, 1, 1, 52) OK
SARIMA(1, 1, 1)(0, 1, 1, 52) OK
SARIMA(1, 1, 1)(1, 1, 0, 52) OK
ARIMAX(1, 1, 0) OK
ARIMAX(0, 1, 1) OK
ARIMAX(1, 1, 1) OK
ARIMAX(2, 1, 1) OK
ARIMAX(1, 1, 2) OK
ARIMAX(2, 1, 2) OK
SARIMAX(1, 1, 1)(1, 1, 1, 52) OK
SARIMAX(1, 1, 0)(1, 1, 1, 52) OK
SARIMAX(0, 1, 1)(1, 1, 1, 52) OK
SARIMAX(2, 1, 1)(1, 1, 1, 52) OK
SARIMAX(1, 1, 2)(1, 1, 1, 52) OK
SARIMAX(1, 1, 1)(0, 1, 1, 52) OK
SARIMAX(1, 1, 1)(1, 1, 0, 52) OK


In [16]:
import os

ruta_excel = os.path.join(
    ubicacion_datos_consolidados_janis,
    'resultados_modelo_corr_rezago_relevante.xlsx'
)

with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:

    resultados_totales = {
        '80_20': pd.concat([resultados_arima_80, resultados_sarima_80, resultados_arimax_80, resultados_sarimax_80], ignore_index=True),
        '90_10': pd.concat([resultados_arima_90, resultados_sarima_90, resultados_arimax_90, resultados_sarimax_90], ignore_index=True),
        '95_5':  pd.concat([resultados_arima_95, resultados_sarima_95, resultados_arimax_95, resultados_sarimax_95], ignore_index=True)
    }

    resultados_totales['80_20'].to_excel(writer, sheet_name='80_20', index=False)
    resultados_totales['90_10'].to_excel(writer, sheet_name='90_10', index=False)
    resultados_totales['95_5'].to_excel(writer, sheet_name='95_5', index=False)

print("Resultados guardados en:", ruta_excel)

Resultados guardados en: C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\datos_consolidados\3_datos_modelos\datos_corr\datos_modelo_1_var_rezago_relevante\resultados\resultados_modelo_corr_rezago_relevante.xlsx
